# 🎮 Proyecto Street Fighter con MediaPipe Hands  

Nuestro proyecto consiste en una **versión interactiva de Street Fighter** que utiliza la tecnología de **MediaPipe Hands** para controlar los movimientos de los personajes mediante gestos con las manos.  

En lugar de utilizar un control tradicional, los jugadores podrán **pelear con movimientos físicos**, haciendo la experiencia más inmersiva y dinámica.  

## ⚙️ ¿Cómo funciona?  
- A partir de estos puntos, el sistema interpreta gestos específicos para transformarlos en acciones dentro del juego.  

## 🕹️ Movimientos implementados  
- ✊ **Golpe** → Movimiento de puño cerrado.  
- ✋ **Avanzar** → Gesto de mano hacia adelante.  
- 🖐️ **Saltar** → Gesto de mano levantada.  
- 🦵 **Patada** → Combinación de gestos que activan la acción de patear.  


In [1]:
import mediapipe as mp
print(mp.__version__)


0.10.21


In [2]:
def limites_saltar_agachar(y_nariz, y_hombro_izq, y_hombro_der):
    y_delta_hombro_nariz = abs(y_nariz - ((y_hombro_izq + y_hombro_der) / 2))
    
    limite_saltar = y_nariz - y_delta_hombro_nariz
    limite_agachar = 1.33 * (y_nariz + y_delta_hombro_nariz)
    
    return limite_saltar, limite_agachar

In [3]:
def limites_acerca_alejar(x_nariz, x_hombro_izq, x_hombro_der):
    x_hombro = (x_hombro_izq + x_hombro_der) / 2
    x_delta_hombro_nariz = abs(x_nariz - (x_hombro))
    
    limite_acercar = 0.8 * (x_hombro + x_delta_hombro_nariz)
    limite_alejar = 1.4 * (x_hombro - x_delta_hombro_nariz)
    
    return limite_acercar, limite_alejar

In [109]:
import cv2
import mediapipe as mp
import random
import time
import pydirectinput

# --- Inicializar MediaPipe Pose (para cuerpo) ---
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing_pose = mp.solutions.drawing_utils

# --- Inicializar MediaPipe Hands (para manos) ---
mp_hands = mp.solutions.hands
hands = mp_hands.Hands()
mp_drawing_hands = mp.solutions.drawing_utils

# --- Helpers de cámara ---
def open_cam(idx):
    cap = cv2.VideoCapture(idx, cv2.CAP_DSHOW)
    if not cap.isOpened():
        return None
    ok, _ = cap.read()
    if not ok:
        cap.release()
        return None
    return cap

# --- Abre cámara inicial (0; si no, 1) ---
current_idx = 0
cap = open_cam(current_idx)
if cap is None:
    current_idx = 0
    cap = open_cam(current_idx)
if cap is None:
    raise RuntimeError("No se pudo abrir cámara 0 ni 1.")
"""
# --- Captura de cámara ---
cap = cv2.VideoCapture(0)
"""
# --- Variables del juego de manos ---
Piedra = 100
Papel = 101
Tijera = 102
texto = "..."
texto_ia = "IA: ..."
resultado = ""
mostrar_resultado = False
tiempo_inicio = 0

# --- Variable para acción corporal ---
accion = "Neutra"

""""y_nariz_inicial = -1 # y0
y_hombro_izq_inicial = -1 # y11
y_hombro_der_inicial = -1 # y12
"""
proporcion_inicial_flag = False

limite_saltar = 0
limite_agachar = 0

limite_acercar = 0
limite_alejar = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Copia de frame original
    img = frame.copy()
    fraccion = 0.5
    image = cv2.resize(img, (0, 0), fx=fraccion, fy=fraccion, interpolation=cv2.INTER_NEAREST)
    image_rgb = cv2.cvtColor(cv2.flip(image, 1), cv2.COLOR_BGR2RGB)

    # Procesar pose y manos
    results_pose = pose.process(image_rgb)
    #results_hands = hands.process(image_rgb)

    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    # Calculo de limite inferior y superior para salto/agacharse
    # (RECORDAR CHECAR SI LOS 3 VALORES SI ESTAN SIENDO DETECTADOS)
    if results_pose.pose_landmarks:

        y_nariz = results_pose.pose_landmarks.landmark[0].y
        y_hombro_izq = results_pose.pose_landmarks.landmark[11].y
        y_hombro_der = results_pose.pose_landmarks.landmark[12].y

        x_nariz = results_pose.pose_landmarks.landmark[0].x
        x_hombro_izq = results_pose.pose_landmarks.landmark[11].x
        x_hombro_der = results_pose.pose_landmarks.landmark[12].x

        #x_hombro = (x_hombro_izq + x_hombro_der) / 2
  
    """nariz_visibility = results_pose.pose_landmarks.landmark[0].visibility
    hombro_izq_visibility = results_pose.pose_landmarks.landmark[11].visibility
    hombro_der_visibility = results_pose.pose_landmarks.landmark[12].visibility
    """

    # (RECORDAR CHECAR SI LOS 3 VALORES SI ESTAN SIENDO DETECTADOS)
    # (RECORDAR CHECAR SI LOS 3 VALORES SI ESTAN SIENDO DETECTADOS)
    # (RECORDAR CHECAR SI LOS 3 VALORES SI ESTAN SIENDO DETECTADOS)




    #if y_nariz_inicial == -1 and y_hombro_izq_inicial == -1 and y_hombro_der_inicial == -1:
    if proporcion_inicial_flag == False:

        limite_saltar, limite_agachar = limites_saltar_agachar(y_nariz, y_hombro_izq, y_hombro_der)
        limite_acercar, limite_alejar = limites_acerca_alejar(x_nariz, x_hombro_izq, x_hombro_der)
        proporcion_inicial_flag = True

        h, w = image_bgr.shape[:2]

        # Jump limit line (green)
        y_js = limite_saltar
        if y_js < 0.0: y_js = 0.0
        if y_js > 1.0: y_js = 1.0
        y_js_px = int(y_js * h)
        cv2.line(image_bgr, (0, y_js_px), (w - 1, y_js_px), (0, 255, 0), 2)

        # Crouch limit line (red)
        y_ag = limite_agachar
        if y_ag < 0.0: y_ag = 0.0
        if y_ag > 1.0: y_ag = 1.0
        y_ag_px = int(y_ag * h)
        cv2.line(image_bgr, (0, y_ag_px), (w - 1, y_ag_px), (0, 0, 255), 2)


    # ==================== DETECCIÓN DE POSE ====================
    if results_pose.pose_landmarks:
        mp_drawing_pose.draw_landmarks(image_bgr, results_pose.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        nariz_y = results_pose.pose_landmarks.landmark[0].y  # coordenada Y de la nariz

        x11 = results_pose.pose_landmarks.landmark[11].x
        x12 = results_pose.pose_landmarks.landmark[12].x
        x_hombro = (x11 + x12) / 2

        if y_nariz < limite_saltar:
            pydirectinput.press('up')
            accion = "Saltar"
            print("Saltar")

        elif y_nariz > limite_agachar:
            pydirectinput.press('down')
            accion = "Agachar"
            print("Agacharse")

        else:
            accion = "Neutra"
            print("Neutro")

        
        print(f"limite1: {limite_acercar}, limite2: {limite_alejar}, hombros: {x_hombro}")

        if x_hombro < limite_acercar:
            pydirectinput.press('left')
            accion = "Alejar"
            print("Alejar")

        elif x_hombro > limite_alejar:
            pydirectinput.press('right')
            accion = "Acercar"
            print("Acercar")

        else:
            accion = "Neutra"
            print("Neutro")

    """
    # ==================== DETECCIÓN DE MANOS ====================
    texto = "..."
    if results_hands.multi_hand_landmarks:
        for h_landmark in results_hands.multi_hand_landmarks:
            mp_drawing_hands.draw_landmarks(
                image_bgr,
                h_landmark, mp_hands.HAND_CONNECTIONS,
                landmark_drawing_spec=mp_drawing_hands.DrawingSpec(color=(0, 0, 150), circle_radius=4, thickness=4),
                connection_drawing_spec=mp_drawing_hands.DrawingSpec(color=(150, 150, 0), thickness=4)
            )

            y0 = h_landmark.landmark[0].y
            y8 = h_landmark.landmark[8].y
            y12 = h_landmark.landmark[12].y
            y16 = h_landmark.landmark[16].y
            y20 = h_landmark.landmark[20].y

            d8 = abs(y8 - y0)
            d12 = abs(y12 - y0)
            d16 = abs(y16 - y0)
            d20 = abs(y20 - y0)

            # ---- Lógica piedra, papel o tijera ----
            if d8 < 0.40 and d12 < 0.40 and d16 < 0.40 and d20 < 0.40:
                texto = "Ctrl"  # Piedra
                pydirectinput.press('right')
            elif d8 > 0.2 and d12 > 0.2 and d16 > 0.2 and d20 > 0.2:
                texto = "Shift"  # Papel
                pydirectinput.press('alt')
            elif d8 > 0.2 and d12 > 0.2 and d16 < 0.40 and d20 < 0.40:
                texto = "Alt"  # Tijera
                pydirectinput.press('x')

                
                
    # ==================== LÓGICA DEL JUEGO ====================
    if key == ord('r'):
        if texto in ["Piedra", "Papel", "Tijera"]:
            jugada_ia = random.randint(100, 102)
            if jugada_ia == 100:
                texto_ia = "IA: Piedra"
            elif jugada_ia == 101:
                texto_ia = "IA: Papel"
            elif jugada_ia == 102:
                texto_ia = "IA: Tijera"

            if (texto == "Piedra" and "Tijera" in texto_ia) or \
               (texto == "Papel" and "Piedra" in texto_ia) or \
               (texto == "Tijera" and "Papel" in texto_ia):
                resultado = "Ganaste!"
            elif (texto == "Piedra" and "Papel" in texto_ia) or \
                 (texto == "Papel" and "Tijera" in texto_ia) or \
                 (texto == "Tijera" and "Piedra" in texto_ia):
                resultado = "Perdiste!"
            else:
                resultado = "Empate!"

            mostrar_resultado = True
            tiempo_inicio = time.time()

    if mostrar_resultado and time.time() - tiempo_inicio > 3:
        texto_ia = "IA: ..."
        resultado = ""
        mostrar_resultado = False


    """

    # ==================== MOSTRAR EN PANTALLA ====================
    image_flipped = cv2.flip(image_bgr, 1)
    cv2.putText(image_flipped, f"Jugador: {texto}", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    cv2.putText(image_flipped, f"Accion cuerpo: {accion}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("Pose + Hand Tracking", image_flipped)


    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4277448207139969
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.42763157933950424
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4276420846581459
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4276764616370201
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4279946982860565
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4282739609479904
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4279423803091049
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.4284735918045044
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.428376279771328
Neutro
Neutro
limite1: 0.39209191799163823, limite2: 0.5115246415138245, hombros: 0.42947107553482